# 02 — Three-Layer Neural Network on MNIST with Keras

**Objective:** Build a feedforward neural network with **three Dense layers** (two hidden + one output) using TensorFlow/Keras on the MNIST handwritten-digit dataset.
We normalize input data, train the model, evaluate it on a held-out test set, and save the trained weights.

---
## 1. Architecture Overview

```
Input image  : 28x28 grayscale  ->  Flatten to 784 features
Dense layer 1: 128 neurons, ReLU activation   (Hidden layer 1)
Dense layer 2: 64  neurons, ReLU activation   (Hidden layer 2)
Dense layer 3: 10  neurons, Softmax activation (Output layer)
```

**Why these layers?**
- `Flatten`: Converts the 2D image (28x28) into a 1D vector of 784 values so Dense layers can process it.
- **Hidden layers with ReLU**: Allow the network to learn non-linear patterns in the data. ReLU (Rectified Linear Unit) avoids the vanishing gradient problem.
- **Output layer with Softmax**: Converts raw scores to probabilities across 10 digit classes. Probabilities sum to 1.

---
## 2. Imports and Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Hyperparameters ---
HIDDEN_1    = 128   # neurons in first hidden layer
HIDDEN_2    = 64    # neurons in second hidden layer
EPOCHS      = 10
BATCH_SIZE  = 64
VAL_SPLIT   = 0.1   # 10% of training data used for validation
MODEL_PATH  = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'mnist_model.keras')
# -----------------------

print(f"TensorFlow version : {tf.__version__}")
print(f"Keras version      : {keras.__version__}")
print(f"Model will be saved: {MODEL_PATH}")

---
## 3. Load and Inspect MNIST

In [ ]:
# Load MNIST from the Keras dataset API
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

print(f"Training samples  : {X_train_raw.shape[0]:,}")
print(f"Test samples      : {X_test_raw.shape[0]:,}")
print(f"Image shape       : {X_train_raw.shape[1:]}")
print(f"Pixel value range : {X_train_raw.min()} - {X_train_raw.max()}")
print(f"Classes           : {sorted(set(y_train.tolist()))}")

---
## 4. Display Sample Images

In [ ]:
# Display a 5x5 grid of sample training images
fig, axes = plt.subplots(5, 10, figsize=(14, 7))
for digit in range(10):
    # Pick 5 examples of each digit
    indices = np.where(y_train == digit)[0][:5]
    for row, idx in enumerate(indices):
        ax = axes[row, digit]
        ax.imshow(X_train_raw[idx], cmap="gray")
        ax.set_title(str(digit), fontsize=9)
        ax.axis("off")
plt.suptitle("MNIST Sample Images (5 per class)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Preprocessing — Normalize Pixel Values

In [ ]:
# Normalize: convert pixel values from [0, 255] to [0.0, 1.0]
# This improves numerical stability and speeds up gradient descent.
X_train = X_train_raw.astype("float32") / 255.0
X_test  = X_test_raw.astype("float32")  / 255.0

print(f"Normalized pixel range (train): {X_train.min():.2f} - {X_train.max():.2f}")
print(f"Normalized pixel range (test) : {X_test.min():.2f}  - {X_test.max():.2f}")

---
## 6. Build the Model

In [ ]:
def build_model(hidden1=128, hidden2=64):
    """Build a 3-Dense-layer feedforward network for MNIST."""
    model = keras.Sequential([
        # Input transformation: flatten 28x28 to 784
        keras.layers.Flatten(input_shape=(28, 28), name="flatten"),
        # Hidden layer 1: learns low-level features
        keras.layers.Dense(hidden1, activation="relu", name="hidden_1"),
        # Hidden layer 2: learns higher-level combinations
        keras.layers.Dense(hidden2, activation="relu", name="hidden_2"),
        # Output layer: 10 classes with softmax (probabilities sum to 1)
        keras.layers.Dense(10, activation="softmax", name="output"),
    ])
    return model

model = build_model(HIDDEN_1, HIDDEN_2)
model.summary()

---
## 7. Compile the Model

In [ ]:
model.compile(
    optimizer="adam",                          # Adam: adaptive learning rate — works well out of the box
    loss="sparse_categorical_crossentropy",    # Correct loss for integer class labels
    metrics=["accuracy"]
)
print("Model compiled successfully.")
print("  Optimizer : Adam")
print("  Loss      : SparseCategoricalCrossentropy")
print("  Metric    : Accuracy")

---
## 8. Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    verbose=1
)

---
## 9. Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")

---
## 10. Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
axes[0].plot(history.history["accuracy"],     label="Train Accuracy",      linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation Accuracy",  linewidth=2, linestyle="--")
axes[0].set_title("Accuracy over Epochs", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# --- Loss ---
axes[1].plot(history.history["loss"],     label="Train Loss",      linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation Loss",  linewidth=2, linestyle="--")
axes[1].set_title("Loss over Epochs", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

---
## 11. Sample Predictions with Confidence Scores

In [ ]:
# Get predictions on test set
probs = model.predict(X_test, verbose=0)     # shape: (10000, 10)
preds = np.argmax(probs, axis=1)

# Display 16 random test samples
sample_idx = np.random.choice(len(X_test), size=16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, idx in zip(axes.ravel(), sample_idx):
    ax.imshow(X_test[idx], cmap="gray")
    pred_label = preds[idx]
    true_label = y_test[idx]
    confidence = probs[idx, pred_label] * 100
    color = "green" if pred_label == true_label else "red"
    ax.set_title(f"Pred: {pred_label}  True: {true_label}\n{confidence:.1f}%",
                 color=color, fontsize=10)
    ax.axis("off")
plt.suptitle("Sample Test Predictions (green=correct, red=wrong)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 12. Save the Trained Model

In [ ]:
# Save in the native Keras format (recommended for TF 2.x)
save_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'mnist_model.keras')
try:
    model.save(save_path)
    print(f"Model saved to: {save_path}")
except Exception as e:
    # Fallback: save in TF SavedModel format
    fallback_path = save_path.replace('.keras', '.h5')
    model.save(fallback_path)
    print(f"Saved as HDF5 fallback: {fallback_path}")
    print(f"Original error: {e}")

---
## 13. Observations and Conclusion

- The model achieves **~97–98% test accuracy** in just 10 epochs, demonstrating that a simple dense network is sufficient for MNIST.
- **Training vs Validation accuracy**: A small gap is normal; a large gap indicates overfitting.
- **ReLU** in hidden layers allows the model to learn non-linear decision boundaries across the 784-dimensional pixel space.
- **Softmax** output correctly normalizes predictions to sum to 1, giving interpretable confidence scores.
- The saved model will be loaded in the interactive digit recognizer (notebook 05).

### Model Summary

| Layer       | Neurons | Activation | Parameters      |
|-------------|---------|------------|-----------------|
| Flatten     | 784     | -          | 0               |
| Dense 1     | 128     | ReLU       | 784*128 + 128   |
| Dense 2     | 64      | ReLU       | 128*64 + 64     |
| Output      | 10      | Softmax    | 64*10 + 10      |